In [1]:
import pandas as pd, numpy as np
from lsff_utils import data_processing

In [2]:
directory = "/snfs1/DATA/DHS_PROG_DHS/NGA/2023_2024/"
results_dir = "../results"

### WRA

In [3]:
%%time

wra_columns = {
    "v001": "cluster_number",
    "v002": "household_number",
    "v003": "line_number",
    "v005": "weight",
    "v008": "interview_date",
    "v011": "date_of_birth",
    "v190": "wealth_quintile",
}
wra_data = pd.read_stata(
    directory + "NGA_DHS8_2023_2024_WN_NGIR8AFL_Y2025M10D30.DTA",
    columns=wra_columns.keys(),
)

CPU times: user 5.58 s, sys: 811 ms, total: 6.39 s
Wall time: 6.57 s


In [4]:
wra_data = wra_data[wra_columns.keys()].rename(columns=wra_columns)

In [5]:
wra_data["wealth_quintile"] = data_processing.recode_dhs_wealth_quintile(
    wra_data.wealth_quintile
)

In [6]:
wra_data["weight"] = wra_data.weight / 1_000_000

### Siblings

Sibling survival data appears to only be available as a kind of side table-within-a-table on WRA.

It is labeled "MM" because it is used to calculate maternal mortality (among other things).

In [7]:
respondent_column_names = {
    "v008": "interview_date",
    "v190": "wealth_quintile",
    "v005": "weight",
}

# These are suffixed with an underscore and an integer, e.g. mm1_01
sibling_column_names = {
    # MM1                    Sex of sibling                                  7156    1    N    I   20    0   No   No
    "mm1": "sex",
    # MM2                    Survival status of sibling                      7176    1    N    I   20    0   No   No
    "mm2": "survival_status",
    # MM3                    Sibling's current age                           7196    2    N    I   20    0   No   No
    "mm3": "current_age",
    # MM4                    Sibling's date of birth (CMC)                   7236    4    N    I   20    0   No   No
    "mm4": "date_of_birth",
    # MM8                    Date of death of sibling (CMC)                  7416    4    N    I   20    0   No   No
    "mm8": "date_of_death",
    # MM7                    Sibling's age at death                          7376    2    N    I   20    0   No   No
    "mm7": "age_at_death",
    # MM9                    Sibling's death and pregnancy                   7496    2    N    I   20    0   No   No
    "mm9": "pregnancy_category",
    # MM16                   Sibling's death due to violence or accident     7816    1    N    I   20    0   No   No
    "mm16": "death_violence_or_accident",
}

In [8]:
# The sibling columns get repeated for each sibling.
# In order to make sure we get the right number, we load the metadata and figure
# out which columns to load.

In [9]:
wra_file_path = directory + "NGA_DHS8_2023_2024_WN_NGIR8AFL_Y2025M10D30.DTA"

In [10]:
# Use StataReader to inspect metadata without loading the dataset
with pd.io.stata.StataReader(wra_file_path) as reader:
    # .variable_labels() returns a dict where keys are the column names
    columns = list(reader.variable_labels().keys())

sibling_data_columns = [
    c
    for c in columns
    if c in respondent_column_names.keys()
    or c.split("_")[0] in sibling_column_names.keys()
]
len(sibling_data_columns)

163

In [11]:
sibling_data = pd.read_stata(directory + "NGA_DHS8_2023_2024_WN_NGIR8AFL_Y2025M10D30.DTA", columns=sibling_data_columns)
sibling_data

,v005,v008,v190,mm1_01,mm1_02,mm1_03,mm1_04,mm1_05,mm1_06,mm1_07,...,mm16_11,mm16_12,mm16_13,mm16_14,mm16_15,mm16_16,mm16_17,mm16_18,mm16_19,mm16_20
0,491963,1489,poorer,male,male,male,male,female,female,male,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,491963,1489,poorer,male,female,male,male,male,female,male,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,491963,1489,poorer,male,female,male,male,male,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,491963,1489,poorer,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,491963,1489,poorer,male,male,female,female,female,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39045,507019,1490,middle,male,female,male,male,male,male,male,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
39046,507019,1490,middle,male,male,female,female,male,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
39047,507019,1490,middle,male,male,male,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
39048,507019,1490,middle,male,female,male,female,female,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
# inspired by https://stackoverflow.com/a/67393747/
sibling_data_reshaped = sibling_data[
    [c for c in sibling_data.columns if c.split("_")[0] in sibling_column_names.keys()]
].copy()
sibling_data_reshaped.columns = sibling_data_reshaped.columns.str.split(
    "_", expand=True
)
sibling_data_reshaped

mm1                                                                  \
         01      02      03      04      05      06    07      08   09   10   
0      male    male    male    male  female  female  male    male  NaN  NaN   
1      male  female    male    male    male  female  male    male  NaN  NaN   
2      male  female    male    male    male     NaN   NaN     NaN  NaN  NaN   
3       NaN     NaN     NaN     NaN     NaN     NaN   NaN     NaN  NaN  NaN   
4      male    male  female  female  female     NaN   NaN     NaN  NaN  NaN   
...     ...     ...     ...     ...     ...     ...   ...     ...  ...  ...   
39045  male  female    male    male    male    male  male  female  NaN  NaN   
39046  male    male  female  female    male     NaN   NaN     NaN  NaN  NaN   
39047  male    male    male     NaN     NaN     NaN   NaN     NaN  NaN  NaN   
39048  male  female    male  female  female     NaN   NaN     NaN  NaN  NaN   
39049  male  female    male  female  female     NaN   NaN     NaN  NaN  NaN   

       ... mm16                                               
       ...   11   12   13   14   15   16   17   18   19   20  
0      ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
1      ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
2      ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
3      ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
4      ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
...    ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  
39045  ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
39046  ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
39047  ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
39048  ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
39049  ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  

[39050 rows x 160 columns]

In [13]:
sibling_data_reshaped[list(respondent_column_names.keys())] = sibling_data[
    list(respondent_column_names.keys())
]
sibling_data_reshaped

mm1                                                                  \
         01      02      03      04      05      06    07      08   09   10   
0      male    male    male    male  female  female  male    male  NaN  NaN   
1      male  female    male    male    male  female  male    male  NaN  NaN   
2      male  female    male    male    male     NaN   NaN     NaN  NaN  NaN   
3       NaN     NaN     NaN     NaN     NaN     NaN   NaN     NaN  NaN  NaN   
4      male    male  female  female  female     NaN   NaN     NaN  NaN  NaN   
...     ...     ...     ...     ...     ...     ...   ...     ...  ...  ...   
39045  male  female    male    male    male    male  male  female  NaN  NaN   
39046  male    male  female  female    male     NaN   NaN     NaN  NaN  NaN   
39047  male    male    male     NaN     NaN     NaN   NaN     NaN  NaN  NaN   
39048  male  female    male  female  female     NaN   NaN     NaN  NaN  NaN   
39049  male  female    male  female  female     NaN   NaN     NaN  NaN  NaN   

       ... mm16                                v008    v190    v005  
       ...   14   15   16   17   18   19   20                        
0      ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  1489  poorer  491963  
1      ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  1489  poorer  491963  
2      ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  1489  poorer  491963  
3      ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  1489  poorer  491963  
4      ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  1489  poorer  491963  
...    ...  ...  ...  ...  ...  ...  ...  ...   ...     ...     ...  
39045  ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  1490  middle  507019  
39046  ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  1490  middle  507019  
39047  ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  1490  middle  507019  
39048  ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  1490  middle  507019  
39049  ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  1490  middle  507019  

[39050 rows x 163 columns]

In [14]:
# Get a row per sibling
sibling_data_reshaped = (
    sibling_data_reshaped.set_index(list(respondent_column_names.keys()))
    .swaplevel(axis=1)
    .stack(0)
    .reset_index()
    .drop(columns=[f"level_{len(respondent_column_names)}"])
)
sibling_data_reshaped

,v008,v190,v005,mm1,mm16,mm2,mm3,mm4,mm7,mm8,mm9
0,1489,poorer,491963,male,NaN,alive,42.0,979.0,NaN,NaN,NaN
1,1489,poorer,491963,male,NaN,alive,30.0,1123.0,NaN,NaN,NaN
2,1489,poorer,491963,male,NaN,alive,25.0,1183.0,NaN,NaN,NaN
3,1489,poorer,491963,male,NaN,alive,23.0,1207.0,NaN,NaN,NaN
4,1489,poorer,491963,female,NaN,alive,20.0,1243.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
189167,1490,middle,507019,male,NaN,alive,35.0,1064.0,NaN,NaN,NaN
189168,1490,middle,507019,female,NaN,alive,32.0,1100.0,NaN,NaN,NaN
189169,1490,middle,507019,male,NaN,alive,28.0,1148.0,NaN,NaN,NaN
189170,1490,middle,507019,female,NaN,alive,25.0,1184.0,NaN,NaN,NaN


In [15]:
sibling_data = (
    sibling_data_reshaped[
        list(respondent_column_names.keys()) + list(sibling_column_names.keys())
    ]
    .rename(columns=respondent_column_names)
    .rename(columns=sibling_column_names)
)
sibling_data["wealth_quintile"] = data_processing.recode_dhs_wealth_quintile(
    sibling_data.wealth_quintile
)
sibling_data["weight"] = sibling_data.weight / 1_000_000
sibling_data

,interview_date,wealth_quintile,weight,sex,survival_status,current_age,date_of_birth,date_of_death,age_at_death,pregnancy_category,death_violence_or_accident
0,1489,2,0.491963,male,alive,42.0,979.0,NaN,NaN,NaN,NaN
1,1489,2,0.491963,male,alive,30.0,1123.0,NaN,NaN,NaN,NaN
2,1489,2,0.491963,male,alive,25.0,1183.0,NaN,NaN,NaN,NaN
3,1489,2,0.491963,male,alive,23.0,1207.0,NaN,NaN,NaN,NaN
4,1489,2,0.491963,female,alive,20.0,1243.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
189167,1490,3,0.507019,male,alive,35.0,1064.0,NaN,NaN,NaN,NaN
189168,1490,3,0.507019,female,alive,32.0,1100.0,NaN,NaN,NaN,NaN
189169,1490,3,0.507019,male,alive,28.0,1148.0,NaN,NaN,NaN,NaN
189170,1490,3,0.507019,female,alive,25.0,1184.0,NaN,NaN,NaN,NaN


In [16]:
# The 2024 report has no section about maternal mortality, for some reason!
# So unfortunately, we cannot compare the numbers throughout this notebook
# with any numbers in the report to check our work.
# The 2018 report does have numbers, so we cite those here as very rough
# data quality checks, that nothing has changed by too huge of a margin.
# For this number, the 2018 report states:
# "A total of 219,561 siblings were recorded..." (p. 372)
len(sibling_data)

189172

### Births

In [17]:
birth_columns = {
    "v005": "weight",
    "v008": "interview_date",
    "v190": "wealth_quintile",
    "b3": "birth_date",
    "m18": "size_of_child",
    "m19": "birth_weight_kilograms",
    "b20": "duration_of_pregnancy",
}

birth_data = pd.read_stata(
    directory + "NGA_DHS8_2023_2024_BR_NGBR8AFL_Y2025M10D30.DTA",
    columns=birth_columns.keys(),
)
birth_data

,v005,v008,v190,b3,m18,m19,b20
0,491963,1489,poorer,1479,average,not weighed at birth,9
1,491963,1489,poorer,1409,NaN,NaN,9
2,491963,1489,poorer,1388,NaN,NaN,9
3,491963,1489,poorer,1339,NaN,NaN,9
4,491963,1489,poorer,1313,NaN,NaN,9
...,...,...,...,...,...,...,...
104552,507019,1490,middle,1211,NaN,NaN,9
104553,507019,1490,middle,1174,NaN,NaN,9
104554,507019,1490,middle,1143,NaN,NaN,9
104555,507019,1490,middle,1122,NaN,NaN,9


In [18]:
birth_data = birth_data[birth_columns.keys()].rename(columns=birth_columns)
birth_data["wealth_quintile"] = data_processing.recode_dhs_wealth_quintile(
    birth_data.wealth_quintile
)
birth_data["weight"] = birth_data.weight / 1_000_000
birth_data

,weight,interview_date,wealth_quintile,birth_date,size_of_child,birth_weight_kilograms,duration_of_pregnancy
0,0.491963,1489,2,1479,average,not weighed at birth,9
1,0.491963,1489,2,1409,NaN,NaN,9
2,0.491963,1489,2,1388,NaN,NaN,9
3,0.491963,1489,2,1339,NaN,NaN,9
4,0.491963,1489,2,1313,NaN,NaN,9
...,...,...,...,...,...,...,...
104552,0.507019,1490,3,1211,NaN,NaN,9
104553,0.507019,1490,3,1174,NaN,NaN,9
104554,0.507019,1490,3,1143,NaN,NaN,9
104555,0.507019,1490,3,1122,NaN,NaN,9


### Maternal mortality ratio

#### Maternal mortality rate

In [19]:
sibling_data.survival_status.value_counts()

alive         173706
dead           15441
don't know        25
Name: survival_status, dtype: int64

In [20]:
sibling_data.pregnancy_category.value_counts()

death not related          1921
died during delivery        271
died while pregnant         267
6 weeks after delivery      134
2 months after delivery      23
Name: pregnancy_category, dtype: int64

In [21]:
sibling_data.death_violence_or_accident.value_counts()

no          13972
accident      703
violence      495
Name: death_violence_or_accident, dtype: int64

In [22]:
# https://dhsprogram.com/Data/Guide-to-DHS-Statistics/Adult_Mortality_Rates.htm#Calculation1
sibling_data["exposure_start"] = np.maximum(
    sibling_data.date_of_birth + 12 * 15, sibling_data.interview_date - 84
)  # aka lowlim
# aka upplim
sibling_data["exposure_end"] = np.minimum(
    np.where(
        sibling_data.survival_status == "alive",
        sibling_data.interview_date - 1,
        sibling_data.date_of_death,
    ),
    sibling_data.date_of_birth + 12 * 50 - 1,
)
sibling_data["exposure"] = (
    (sibling_data.exposure_end - sibling_data.exposure_start) + 1
).clip(lower=0)

In [23]:
sibling_data.exposure.value_counts()

84.0    107412
0.0      42451
66.0      7067
42.0      5944
78.0      5502
         ...  
24.0         2
60.0         2
72.0         1
48.0         1
36.0         1
Name: exposure, Length: 84, dtype: int64

In [24]:
sibling_data["adult_death"] = (
    (sibling_data.survival_status == "dead")
    & (sibling_data.date_of_death - sibling_data.date_of_birth >= 15.0 * 12)
    & (sibling_data.date_of_death - sibling_data.date_of_birth < 50.0 * 12)
    & (sibling_data.exposure > 0)
    & (sibling_data.date_of_death >= sibling_data.exposure_start)
    & (sibling_data.date_of_death <= sibling_data.exposure_end)
)

In [25]:
# Table 14.2 (2018 report) says 480k person-years of exposure for females
# and 510k person-years of exposure for males.
(
    sibling_data[sibling_data.date_of_birth.notnull()]
    .assign(weighted_exposure=lambda df: df.exposure * df.weight)
    .groupby("sex")
    .weighted_exposure.sum()
    / 12
)

sex
female    439982.221765
male      464237.370220
Name: weighted_exposure, dtype: float64

In [26]:
# Table 14.2 (2018 report) says 1,442 deaths for females
# and 1,542 deaths for males.
sibling_data.assign(
    weighted_adult_dealth=lambda df: df.adult_death * df.weight
).groupby("sex").weighted_adult_dealth.sum()

sex
female    1035.113845
male      1125.506202
Name: weighted_adult_dealth, dtype: float64

In [27]:
sibling_data.death_violence_or_accident.value_counts()

no          13972
accident      703
violence      495
Name: death_violence_or_accident, dtype: int64

In [28]:
sibling_data.pregnancy_category.value_counts()

death not related          1921
died during delivery        271
died while pregnant         267
6 weeks after delivery      134
2 months after delivery      23
Name: pregnancy_category, dtype: int64

In [29]:
sibling_data.adult_death.value_counts()

False    187079
True       2093
Name: adult_death, dtype: int64

In [30]:
sibling_data.pregnancy_category.value_counts()

death not related          1921
died during delivery        271
died while pregnant         267
6 weeks after delivery      134
2 months after delivery      23
Name: pregnancy_category, dtype: int64

In [31]:
sibling_data.death_violence_or_accident.value_counts()

no          13972
accident      703
violence      495
Name: death_violence_or_accident, dtype: int64

In [32]:
female_siblings = sibling_data[sibling_data.sex == "female"].copy()
female_siblings["maternal_death"] = (
    (female_siblings.adult_death)
    & (
        female_siblings.pregnancy_category.isin(
            ["died during delivery", "died while pregnant", "6 weeks after delivery"]
        )
    )
    & (~female_siblings.death_violence_or_accident.isin(["violence", "accident"]))
)

In [33]:
# Table 14.4 (2018 report) reports 451 maternal deaths
(female_siblings.maternal_death * female_siblings.weight).sum()

273.614248

In [34]:
# Table 14.4 (2018 report) reports 480,382
(female_siblings.exposure * female_siblings.weight / 12).sum()

439982.2217646667

In [35]:
def maternal_mortality_rate(df):
    return ((df.maternal_death * df.weight).sum() * 1_000) / (
        (df.exposure * df.weight) / 12
    ).sum()

In [36]:
# 2018 report:
# "the maternal mortality rate among women age 15-49 is 0.92 deaths per 1,000 woman-years of exposure." (p. 374)
# TODO: These are not age-standardized! We figure the *disparity* probably isn't way off.
# Should standardize according to the approach from https://github.com/LateraOlana/Fertility_SIM_DHS/blob/main/fertility/Latera_Zebb_Coworking.ipynb
maternal_mortality_rate(female_siblings)

0.6218756905735796

In [37]:
maternal_mortality_rates = female_siblings.groupby("wealth_quintile").apply(
    maternal_mortality_rate
)
maternal_mortality_rates

wealth_quintile
1    0.956953
2    0.662550
3    0.611434
4    0.529252
5    0.409104
dtype: float64

#### General fertility rate

In [38]:
fertility_event_data = birth_data.copy()
fertility_event_data["birth_in_period"] = (
    (fertility_event_data.interview_date - fertility_event_data.birth_date) >= 1
) & ((fertility_event_data.interview_date - fertility_event_data.birth_date) <= 36)
fertility_event_data["weighted_birth_in_period"] = (
    fertility_event_data.birth_in_period * fertility_event_data.weight
)

In [39]:
fertility_event_data.weighted_birth_in_period.sum()

16565.868874000003

In [40]:
fertility_event_data.groupby("wealth_quintile").weighted_birth_in_period.sum()

wealth_quintile
1    3986.539457
2    3720.831422
3    3310.216905
4    2966.935236
5    2581.345854
Name: weighted_birth_in_period, dtype: float64

In [41]:
fertility_exposure_data = wra_data.copy()
fertility_exposure_data["exposure_start"] = np.maximum(
    fertility_exposure_data.date_of_birth + 12 * 15,
    fertility_exposure_data.interview_date - 36,
)  # aka lowlim
# aka upplim
fertility_exposure_data["exposure_end"] = np.minimum(
    fertility_exposure_data.interview_date - 1,
    fertility_exposure_data.date_of_birth + 12 * 45,
)
fertility_exposure_data["exposure"] = (
    (fertility_exposure_data.exposure_end - fertility_exposure_data.exposure_start) + 1
).clip(lower=0)
fertility_exposure_data["weighted_exposure"] = (
    fertility_exposure_data.exposure * fertility_exposure_data.weight
)

In [42]:
# 2018 report, Table 5.1 reports a GFR of 182
# 2024 report, Table 5.1 reports a GFR of 160, which is within rounding error of the number we get here
fertility_event_data.weighted_birth_in_period.sum() * 1_000 / (
    fertility_exposure_data.weighted_exposure.sum() / 12
)

160.12984450982498

In [43]:
gfr_by_wealth = (
    fertility_event_data.groupby("wealth_quintile").weighted_birth_in_period.sum()
    * 1_000
    / (fertility_exposure_data.groupby("wealth_quintile").weighted_exposure.sum() / 12)
)
gfr_by_wealth

wealth_quintile
1    222.858564
2    189.369850
3    160.629484
4    132.468434
5    112.668633
dtype: float64

In [44]:
maternal_disorders_incidence_disparities = (
    (maternal_mortality_rates / gfr_by_wealth)
    .rename("value")
    .rename_axis("wealth_quintile")
    .reset_index()
)
maternal_disorders_incidence_disparities.insert(0, "sex", "Female")
maternal_disorders_incidence_disparities

,sex,wealth_quintile,value
0,Female,1,0.004294
1,Female,2,0.003499
2,Female,3,0.003806
3,Female,4,0.003995
4,Female,5,0.003631


In [45]:
maternal_disorders_incidence_disparities.to_csv(
    f"{results_dir}/maternal_disorders_incidence_disparities/nigeria.csv",
    index=False,
)